# HAM10000 期末 04：系統原型驗證與 Grad-CAM 三種案例

不需要 GPU，一般執行階段即可。必須先完成 02，並把原型的三個檔案放到雲端硬碟（見第 1 節）。

**這份做三件事**
1. **驗證原型**：用原型的程式重新預測種子 42 的整份測試集，確認結果和 02 存下的預測一致，證明原型用的是同一個模型、同樣的前處理。
2. **Grad-CAM 三種案例**：正確、漏判、誤報各 4 張，存成新圖 fig_gradcam_melanoma_v2.png，不覆蓋原本的圖。
3. **試用原型**：產生一個暫時的公開網址（約 72 小時有效），可以先在瀏覽器試用。

不會改動 02 的任何結果檔。

## 1. 設定

執行前，請先在雲端硬碟的 ham10000_project 裡建立 **prototype** 資料夾，放入 Claude 給的 app.py、links.json、requirements.txt。模型檔不用複製，程式會直接讀 results_final_cnn/models 裡的檔案。

In [ ]:
import os, sys, subprocess
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/ham10000_project'
PROTO = os.path.join(BASE, 'prototype')
MODEL = os.path.join(BASE, 'results_final_cnn', 'models', 'model__cw-aug__lesion__seed42.pt')
PRED = os.path.join(BASE, 'results_final_cnn', 'predictions', 'prob__cw-aug__lesion__seed42.npz')
SPLIT = os.path.join(BASE, 'results_final_baseline', 'split_assignment.csv')
OUT_FIG = os.path.join(BASE, 'results_final_cnn', 'fig_gradcam_melanoma_v2.png')
for p in [os.path.join(PROTO, 'app.py'), os.path.join(PROTO, 'links.json'), MODEL, PRED, SPLIT]:
    assert os.path.exists(p), f'找不到：{p}'
print('檔案都在')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gradio'], check=True)
os.environ['MODEL_PATH'] = MODEL
sys.path.insert(0, PROTO)
import app
print('原型載入完成')

## 2. 下載 HAM10000 影像

和 00、02 一樣用 kagglehub 下載，第一次約需幾分鐘。

In [ ]:
import kagglehub
kag = kagglehub.dataset_download('kmader/skin-cancer-mnist-ham10000')
HAM = {f[:-4]: os.path.join(r, f) for r, _, fs in os.walk(kag) for f in fs if f.endswith('.jpg') and f.startswith('ISIC_')}
print('HAM10000 影像數：', len(HAM))
assert len(HAM) == 10015

## 3. 驗證原型與 02 的預測一致

02 在 GPU 上以半精度計算，原型在 CPU 上以單精度計算，機率會有很小的差異。判定標準：每張影像七類機率的最大差異在 0.02 以內，且預測類別一致的比例達 99% 以上。

In [ ]:
import numpy as np, pandas as pd, time
from PIL import Image
z = np.load(PRED)
ids = z['test_image_id'].astype(str)
ref = z['test']
probs = np.zeros_like(ref)
t0 = time.time()
for i, iid in enumerate(ids):
    _, x = app.preprocess(Image.open(HAM[iid]))
    probs[i] = app.predict_probs(x)
print(f'{len(ids)} 張，耗時 {time.time() - t0:.0f} 秒')
diff = np.abs(probs - ref).max(1)
agree = (probs.argmax(1) == ref.argmax(1)).mean()
print(f'最大機率差異：{diff.max():.4f}（中位數 {np.median(diff):.5f}）')
print(f'預測類別一致比例：{agree:.4f}')
assert diff.max() < 0.02 and agree >= 0.99, '原型與 02 的預測不一致，請停下來告訴組長'
print('驗證通過：原型與 02 使用同一個模型與前處理')

demo_ids = {'ISIC_0026086': '示範一（黑色素瘤）', 'ISIC_0026077': '示範二（痣）', 'ISIC_0025316': '示範三（黑色素瘤）'}
rows = []
for iid, name in demo_ids.items():
    k = list(ids).index(iid)
    rows.append({'影像': iid, '用途': name, '原型 黑色素瘤機率': round(float(probs[k, app.MEL]), 3), '02 黑色素瘤機率': round(float(ref[k, app.MEL]), 3),
                 '原型 預測類別': app.CLASS_ZH[app.CLASS_ORDER[probs[k].argmax()]]})
print(pd.DataFrame(rows).to_string(index=False))

## 4. Grad-CAM：正確、漏判、誤報

選圖方式與 02 相同（亂數種子 42），所以正確與漏判的 8 張和原本的圖一樣，另外加上 4 張誤報（不是黑色素瘤，卻被判為黑色素瘤）。

In [ ]:
import matplotlib.pyplot as plt
CLS = app.CLASS_ORDER; MEL = app.MEL
sa = pd.read_csv(SPLIT).set_index('image_id')
y = np.array([CLS.index(d) for d in sa.loc[ids, 'dx']])
pred = ref.argmax(1)
rng = np.random.default_rng(42)
is_mel = y == MEL
correct = rng.permutation(ids[is_mel & (pred == MEL)])[:4]
missed = rng.permutation(ids[is_mel & (pred != MEL)])[:4]
false_pos = rng.permutation(ids[(~is_mel) & (pred == MEL)])[:4]
pos = {iid: k for k, iid in enumerate(ids)}

fig, axes = plt.subplots(3, 4, figsize=(13, 10.5))
for ax in axes.ravel():
    ax.axis('off')
for r, (name, group) in enumerate([('Correct', correct), ('Missed', missed), ('False positive', false_pos)]):
    for c, iid in enumerate(group):
        arr, x = app.preprocess(Image.open(HAM[iid]))
        k = pos[iid]
        target = MEL if name != 'Missed' else int(pred[k])
        cam = app.grad_cam(x, target)
        axes[r, c].imshow(arr); axes[r, c].imshow(cam, cmap='jet', alpha=0.4)
        if name == 'Correct':
            t = 'Correct (CAM: mel)'
        elif name == 'Missed':
            t = f'Missed → {CLS[target]} (CAM: {CLS[target]})'
        else:
            t = f'False positive: true {CLS[y[k]]} (CAM: mel)'
        axes[r, c].set_title(f'{iid}\n{t}', fontsize=8)
plt.suptitle('Grad-CAM for melanoma, seed 42 (top: correct, middle: missed, bottom: false positive)')
plt.tight_layout(); plt.savefig(OUT_FIG, dpi=200); plt.show()
print('已存檔：', OUT_FIG)
print('誤報案例的真實類別：', [CLS[y[pos[i]]] for i in false_pos])

## 5. 試用原型

執行後會出現一個 https://……gradio.live 的網址，約 72 小時有效，可以先用瀏覽器試用。要停止就按這一格左邊的停止鍵。正式展示用的網址在 Hugging Face 上，見組長的部署步驟。

In [ ]:
app.demo.launch(share=True, debug=True)